# Busca semantica de cartas (Etapa 3)

*(Feito com ClaudeAI)*

Usa os embeddings gerados na etapa 2 pra montar uma funcao de busca de
verdade: consulta em linguagem natural -> vetor -> similaridade de
cosseno -> top-k cartas, com nome/tipo/custo/texto formatados pra leitura,
nao so uma lista crua de scores.

## Configuração

In [1]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import os, getpass
from openai import OpenAI         
from filtros_mtg import parse_query_llm, parse_query_syntax, hybrid_search, set_all_indexes

In [2]:
meta = json.load(open("Dados/Etapa2_embeddings/embeddings_qwen3_meta.json", encoding="utf-8"))

CARD_DOCUMENTS_PATH = "Dados/Etapa1_card_expanded/card_documents.jsonl"
EMBEDDINGS_PATH = "Dados/Etapa2_embeddings/embeddings_qwen3.npy"
MODEL_NAME = meta["model_id"]           # "Qwen/Qwen3-Embedding-0.6B" (ou a pasta local, se estiver offline)
QUERY_PREFIX = meta["query_prefix"]     # "Instruct: ...\nQuery:"

## Carregar embeddings e dados completos da carta

Os embeddings (etapa 2) foram gerados na mesma ordem em que
`card_documents.jsonl` lista as cartas -- filtrando so as que tem
`embedding_text` (mesmo criterio usado pra gerar os vetores).
Reconstruimos essa lista de nomes direto do arquivo, sem depender de um
segundo arquivo de metadados separado (mais facil de esquecer de enviar
junto e desnecessario, ja que a ordem e' sempre a mesma).

In [3]:
embeddings = np.load(EMBEDDINGS_PATH)

cards = []
with open(CARD_DOCUMENTS_PATH, encoding="utf-8") as f:
    for line in f:
        card = json.loads(line)
        if not card.get("embedding_text"):
            continue
        cards.append(card)

assert len(cards) == embeddings.shape[0], (
    f"Descompasso: {len(cards)} cartas vs {embeddings.shape[0]} vetores -- "
    "regere os embeddings (etapa 2)."
)

# Os filtros por keywords, tags e tipos/subtipos são derivados do mesmo corpus da busca.
set_all_indexes(cards)


names = [c["name"] for c in cards]
cards_by_name = {}
for c in cards:
    cards_by_name.setdefault(c["name"], c)

print(f"{len(cards)} vetores carregados, shape {embeddings.shape}")

33147 vetores carregados, shape (33147, 1024)


## Carregar o modelo

Mesmo modelo da etapa 2 -- precisa ser o mesmo, porque vetores gerados
por modelos diferentes nao sao comparaveis entre si.

In [4]:
model = SentenceTransformer(MODEL_NAME, processor_kwargs={"padding_side": "left"})
model.max_seq_length = meta["max_seq_length"]
print(f"Modelo carregado: {MODEL_NAME}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Modelo carregado: Qwen/Qwen3-Embedding-0.6B


## Funcao de busca

`search(query, k)` embeda a consulta, calcula similaridade de
cosseno contra os 33 mil vetores de uma vez (produto de matriz, rapido o
suficiente sem precisar de indice tipo FAISS nesse tamanho de dataset) e
devolve os top-k com os dados completos da carta.

In [5]:
def search(query: str, k: int = 10):
    query_vec = model.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
    scores = embeddings @ query_vec
    top_k = np.argsort(-scores)[:k]

    results = []
    for i in top_k:
        name = names[i]
        card = cards_by_name.get(name)
        if card is None:
            continue
        results.append({
            "score": float(scores[i]),
            "name": name,
            "type_line": card.get("type_line"),
            "mana_cost": card.get("mana_cost"),
            "oracle_text": card.get("oracle_text"),
        })
    return results


def print_results(results):
    for r in results:
        print(f"{r['score']:.3f}  {r['name']}  ({r['mana_cost'] or '-'})")
        print(f"       {r['type_line']}")
        if r["oracle_text"]:
            print(f"       {r['oracle_text'][:120]}")
        print()

In [6]:
def search2(query, k=10):
    return hybrid_search(query, model, embeddings, cards, QUERY_PREFIX, k=k)

In [7]:
token = os.environ.get("ILUMA_TOKEN") or getpass.getpass("Token da IlumA (sk-...): ")
cliente = OpenAI(base_url="https://iluma.cnpem.br:4000/v1", api_key=token,
                 timeout=180.0, max_retries=1)

def parse_query_iluma(query):
    return parse_query_llm(query, cliente, model="iluma")


def search3(query, k=10):
    return hybrid_search(query, model, embeddings, cards, QUERY_PREFIX, k=k,
                         parser=parse_query_iluma)


Token da IlumA (sk-...):  ········


In [8]:
def search4(query, k=10):
    # sem LLM: o usuario escreve os filtros na sintaxe o:/t:/n:/c:/cmc:/pow:/tou:/loy:/def:/
    # k:/pm:/tag:/cost: (ver comentario de parse_query_syntax em filtros_mtg.py pra lista
    # completa). O que sobrar fora desses tokens vai pro embedding como de costume.
    return hybrid_search(query, model, embeddings, cards, QUERY_PREFIX, k=k,
                         parser=parse_query_syntax)

## Testando com algumas consultas

In [ ]:
for query in [
    "creature that destroys artifacts",
    "cheap green mana creature",
    "counter target spell",
    "deals damage to any target",
]:
    # print(f"\n=== BASE: {query!r} ===")
    # print_results(search(query, k=5))
    # print(f"=== HIBRIDO: {query!r} ===")
    # print_results(search2(query, k=5))
    print(f"=== HIBRIDO+ILUMA: {query!r} ===")
    print_results(search3(query, k=5))

=== HIBRIDO+ILUMA: 'creature that destroys artifacts' ===
(LLM falhou: APITimeoutError; usando o parser de regras)
filtros: {'types': ['creature']} | texto semantico: 'destroys artifacts' | 18576/33147 cartas passam
0.697  Tuktuk Scrapper  ({3}{R})
       Creature — Goblin Artificer Ally
       Whenever this creature or another Ally you control enters, you may destroy target artifact. If that artifact is put into

0.656  Depthshaker Titan  ({5}{R}{R})
       Artifact Creature — Robot
       When this creature enters, any number of target noncreature artifacts you control become 3/3 artifact creatures. Sacrifi

0.646  Mephitic Ooze  ({4}{B})
       Creature — Ooze
       This creature gets +1/+0 for each artifact you control.
Whenever this creature deals combat damage to a creature, destro

0.640  Shrapnel Slinger  ({1}{R})
       Artifact Creature — Phyrexian Beast
       When this creature enters, you may sacrifice a creature. When you do, destroy target artifact an opponent controls.

## Sua consulta

Troque o texto abaixo e rode a celula quantas vezes quiser.

In [10]:
print_results(search4("t:curse", k=10))

filtros: {'types': ['curse']} | texto semantico: '(nenhum)' | 42/33147 cartas passam
0.551  Curse of Misfortunes  ({4}{B})
       Enchantment — Aura Curse
       Enchant player
At the beginning of your upkeep, you may search your library for a Curse card that doesn't have the same 

0.521  Curse of Surveillance  ({4}{U})
       Enchantment — Aura Curse
       Enchant player
At the beginning of enchanted player's upkeep, any number of target players other than that player each d

0.521  Curse of Thirst  ({4}{B})
       Enchantment — Aura Curse
       Enchant player
At the beginning of enchanted player's upkeep, this Aura deals damage to that player equal to the number 

0.502  Curse of Fool's Wisdom  ({4}{B}{B})
       Enchantment — Aura Curse
       Enchant player
Whenever enchanted player draws a card, they lose 2 life and you gain 2 life.
Madness {3}{B} (If you disc

0.475  Curse of the Restless Dead  ({2}{B})
       Enchantment — Aura Curse
       Enchant player
Whenever a land ench

In [11]:
print_results(search3("When creature enters, remove land", k=10))

filtros: nenhum | texto semantico: 'removes land when creature enters' | 33147/33147 cartas passam
0.794  Sundering Titan  ({8})
       Artifact Creature — Golem
       When this creature enters or leaves the battlefield, choose a land of each basic land type, then destroy those lands.

0.763  Ravaging Horde  ({3}{R}{R})
       Creature — Human Soldier
       When this creature enters, destroy target land.

0.763  Goblin Settler  ({3}{R})
       Creature — Goblin
       When this creature enters, destroy target land.

0.763  Ogre Arsonist  ({4}{R})
       Creature — Ogre
       When this creature enters, destroy target land.

0.757  Keldon Firebombers  ({3}{R}{R})
       Creature — Human Soldier
       When this creature enters, each player sacrifices all lands they control except for three.

0.757  Glowing Anemone  ({3}{U})
       Creature — Jellyfish Beast
       When this creature enters, you may return target land to its owner's hand.

0.756  Avenger of Zendikar  ({5}{G}{G})
      

In [19]:
tok = model.tokenizer
print(tok.tokenize("indestructible"), tok.tokenize("destroys"))

card = cards_by_name["Darksteel Brute"]   # troque por uma carta Indestructible que apareceu no seu resultado
txt = card["embedding_text"]
print(txt)
print(len(tok(txt)["input_ids"]), "tokens (limite:", model.max_seq_length, ")")

['inde', 'struct', 'ible'] ['dest', 'roys']
Oracle Text:
Indestructible (Damage and effects that say "destroy" don't destroy this artifact.) 3 generic mana : This artifact becomes a 2/2 Beast artifact creature until end of turn.
Keyword Definitions:
- Indestructible: Indestructible is a static ability. A permanent with indestructible cant be destroyed. Such permanents arent destroyed by lethal damage, and they ignore the state-based action that checks for lethal damage. Multiple instances of indestructible on the same permanent are redundant.
Mana Cost: 2 generic mana
Type: Artifact
Name: Darksteel Brute
123 tokens (limite: 1024 )


In [20]:
import re

query = "creature that destroys artifacts"
q = model.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]

cands = search(query, k=30)
names_c = [r["name"] for r in cands]

def no_reminder(t): return re.sub(r"\s*\([^)]*\)", "", t)
def no_kwdef(t):    return re.sub(r"Keyword Definitions:\n(?:- .*\n)+", "", t)

variants = {
    "original": lambda t: t,
    "sem reminder text": no_reminder,
    "sem definicoes de keyword": no_kwdef,
    "sem os dois": lambda t: no_kwdef(no_reminder(t)),
}

for vname, fn in variants.items():
    docs = [fn(cards_by_name[n]["embedding_text"]) for n in names_c]
    sc = model.encode(docs, normalize_embeddings=True) @ q
    print(f"=== {vname} ===")
    for i in np.argsort(-sc)[:10]:
        print(f"  {sc[i]:.3f}  {names_c[i]}  |  {cards_by_name[names_c[i]]['type_line']}")
    print()

=== original ===
  0.698  Tuktuk Scrapper  |  Creature — Goblin Artificer Ally
  0.690  Darksteel Forge  |  Artifact
  0.672  Structural Assault  |  Sorcery
  0.671  Crush  |  Instant
  0.669  Stone by Sunlight  |  Instant
  0.668  Crumble  |  Instant
  0.665  Gleeful Demolition  |  Sorcery
  0.660  Tidy Conclusion  |  Instant
  0.660  Destructive Tampering  |  Sorcery
  0.658  Depthshaker Titan  |  Artifact Creature — Robot

=== sem reminder text ===
  0.696  Tuktuk Scrapper  |  Creature — Goblin Artificer Ally
  0.671  Crush  |  Instant
  0.671  Crumble  |  Instant
  0.671  Structural Assault  |  Sorcery
  0.665  Gleeful Demolition  |  Sorcery
  0.660  Tidy Conclusion  |  Instant
  0.657  Unlicensed Disintegration  |  Instant
  0.657  Destructive Tampering  |  Sorcery
  0.655  Decimate  |  Sorcery
  0.654  Detonate  |  Sorcery

=== sem definicoes de keyword ===
  0.698  Tuktuk Scrapper  |  Creature — Goblin Artificer Ally
  0.690  Darksteel Forge  |  Artifact
  0.672  Structural Assa

In [ ]:
# search4: mesmos filtros de search3, mas sem depender da IlumA
print_results(search4("t:dinosaur o:create tokens", k=10))